In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 【读取步骤】
# 读取最终合并后的 Train/Test 原始特征表
# ============================================================

model_train = pd.read_parquet(
    "data/processed/model_train_raw_features.parquet"
)

model_test = pd.read_parquet(
    "data/processed/model_test_raw_features.parquet"
)


print(
    "model_train Shape:",
    model_train.shape
)

print(
    "model_test Shape:",
    model_test.shape
)

print(
    "训练集重复客户数量:",
    model_train["SK_ID_CURR"].duplicated().sum()
)

print(
    "测试集重复客户数量:",
    model_test["SK_ID_CURR"].duplicated().sum()
)

print(
    "训练集包含 TARGET:",
    "TARGET" in model_train.columns
)

print(
    "测试集包含 TARGET:",
    "TARGET" in model_test.columns
)

model_train Shape: (307511, 451)
model_test Shape: (48744, 450)
训练集重复客户数量: 0
测试集重复客户数量: 0
训练集包含 TARGET: True
测试集包含 TARGET: False


In [3]:
# ============================================================
# 【检查步骤 1】
# 检查 Train/Test 字段是否一致
# ============================================================

train_feature_columns = [
    col
    for col in model_train.columns
    if col != "TARGET"
]


print(
    "Train/Test 字段名称和顺序是否一致:",
    train_feature_columns
    == model_test.columns.tolist()
)


train_only_columns = sorted(
    set(train_feature_columns)
    - set(model_test.columns)
)

test_only_columns = sorted(
    set(model_test.columns)
    - set(train_feature_columns)
)


print(
    "训练集独有字段:",
    train_only_columns
)

print(
    "测试集独有字段:",
    test_only_columns
)

Train/Test 字段名称和顺序是否一致: True
训练集独有字段: []
测试集独有字段: []


In [4]:
# ============================================================
# 【检查步骤 2】
# 汇总训练集缺失值
# ============================================================

train_missing_summary = (
    model_train
    .drop(columns=["TARGET"])
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


train_missing_summary["missing_rate"] = (
    train_missing_summary["missing_count"]
    / len(model_train)
)


train_missing_summary = (
    train_missing_summary[
        train_missing_summary["missing_count"] > 0
    ]
)


train_missing_summary.head(50)

,missing_count,missing_rate
CC_RECENT_AVG_MIN_PAYMENT_RATIO,270000,0.878017
CC_RECENT_MIN_PAYMENT_RATIO,270000,0.878017
CC_MIN_PAYMENT_RATIO,248232,0.807230
CC_AVG_ACCOUNT_MIN_PAYMENT_RATIO,248232,0.807230
CC_TOTAL_LATEST_UTILIZATION_RATIO,241691,0.785959
CC_RECENT_MAX_UTILIZATION_RATIO,236190,0.768070
CC_RECENT_AVG_UTILIZATION_RATIO,236190,0.768070
CC_AVG_ACCOUNT_UTILIZATION_RATIO,221475,0.720218
CC_MAX_UTILIZATION_RATIO,221475,0.720218
CC_TOTAL_PAYMENT_AMOUNT,220606,0.717392


In [5]:
# ============================================================
# 【检查步骤 3】
# 汇总测试集缺失值
# ============================================================

test_missing_summary = (
    model_test
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


test_missing_summary["missing_rate"] = (
    test_missing_summary["missing_count"]
    / len(model_test)
)


test_missing_summary = (
    test_missing_summary[
        test_missing_summary["missing_count"] > 0
    ]
)


test_missing_summary.head(50)

,missing_count,missing_rate
CC_RECENT_AVG_MIN_PAYMENT_RATIO,42000,0.861645
CC_RECENT_MIN_PAYMENT_RATIO,42000,0.861645
CC_MIN_PAYMENT_RATIO,38127,0.782189
CC_AVG_ACCOUNT_MIN_PAYMENT_RATIO,38127,0.782189
CC_TOTAL_LATEST_UTILIZATION_RATIO,35577,0.729874
CC_RECENT_AVG_UTILIZATION_RATIO,35184,0.721812
CC_RECENT_MAX_UTILIZATION_RATIO,35184,0.721812
COMMONAREA_MEDI,33495,0.687161
COMMONAREA_MODE,33495,0.687161
COMMONAREA_AVG,33495,0.687161


In [6]:
# ============================================================
# 【检查步骤 4】
# 比较 Train/Test 缺失率
# ============================================================

missing_rate_comparison = (
    train_missing_summary[
        ["missing_rate"]
    ]
    .rename(
        columns={
            "missing_rate": "train_missing_rate"
        }
    )
    .join(
        test_missing_summary[
            ["missing_rate"]
        ].rename(
            columns={
                "missing_rate": "test_missing_rate"
            }
        ),
        how="outer"
    )
)


missing_rate_comparison[
    "missing_rate_difference"
] = (
    missing_rate_comparison[
        "test_missing_rate"
    ]
    - missing_rate_comparison[
        "train_missing_rate"
    ]
)


missing_rate_comparison[
    "abs_missing_rate_difference"
] = (
    missing_rate_comparison[
        "missing_rate_difference"
    ].abs()
)


missing_rate_comparison.sort_values(
    "abs_missing_rate_difference",
    ascending=False
).head(50)

,train_missing_rate,test_missing_rate,missing_rate_difference,abs_missing_rate_difference
BB_AVG_SEVERE_DPD_MONTH_RATIO,0.700073,0.131975,-0.568097,0.568097
BB_MAX_HISTORY_MONTHS,0.700073,0.131975,-0.568097,0.568097
BB_TOTAL_MONTH_COUNT,0.700073,0.131975,-0.568097,0.568097
BB_TOTAL_DPD_MONTH_COUNT,0.700073,0.131975,-0.568097,0.568097
BB_MAX_SEVERE_DPD_MONTH_RATIO,0.700073,0.131975,-0.568097,0.568097
BB_MAX_SEVERE_DPD_MONTH_COUNT,0.700073,0.131975,-0.568097,0.568097
BB_AVG_CLOSED_MONTH_RATIO,0.700073,0.131975,-0.568097,0.568097
BB_AVG_DPD_MONTH_COUNT,0.700073,0.131975,-0.568097,0.568097
BB_AVG_DPD_MONTH_RATIO,0.700073,0.131975,-0.568097,0.568097
BB_AVG_HISTORY_MONTHS,0.700073,0.131975,-0.568097,0.568097


In [7]:
# ============================================================
# 【清洗步骤 1】
# 找出 Train/Test 中没有区分能力的常量字段
# ============================================================

# 1. 训练集特征，不包含 TARGET
train_features_only = model_train.drop(
    columns=["TARGET"]
)


# 2. 找出训练集中只有一个非缺失取值的字段
train_constant_columns = [
    col
    for col in train_features_only.columns
    if train_features_only[col].nunique(
        dropna=False
    ) <= 1
]


# 3. 找出测试集中只有一个非缺失取值的字段
test_constant_columns = [
    col
    for col in model_test.columns
    if model_test[col].nunique(
        dropna=False
    ) <= 1
]


# 4. 只删除 Train/Test 都是常量的字段
constant_columns_to_drop = sorted(
    set(train_constant_columns)
    & set(test_constant_columns)
)


print(
    "训练集常量字段数量:",
    len(train_constant_columns)
)

print(
    "测试集常量字段数量:",
    len(test_constant_columns)
)

print(
    "Train/Test共同常量字段数量:",
    len(constant_columns_to_drop)
)

print(
    "准备删除的常量字段:"
)

constant_columns_to_drop

训练集常量字段数量: 10
测试集常量字段数量: 22
Train/Test共同常量字段数量: 10
准备删除的常量字段:


['CODE_GENDER_MISSING',
 'FLAG_OWN_CAR_MISSING',
 'FLAG_OWN_REALTY_MISSING',
 'NAME_CONTRACT_TYPE_MISSING',
 'NAME_EDUCATION_TYPE_MISSING',
 'NAME_FAMILY_STATUS_MISSING',
 'NAME_HOUSING_TYPE_MISSING',
 'NAME_INCOME_TYPE_MISSING',
 'ORGANIZATION_TYPE_MISSING',
 'WEEKDAY_APPR_PROCESS_START_MISSING']

In [8]:
# ============================================================
# 【删除步骤】
# 同时从 Train 和 Test 删除常量字段
# ============================================================

model_train_clean = model_train.drop(
    columns=constant_columns_to_drop
).copy()

model_test_clean = model_test.drop(
    columns=constant_columns_to_drop
).copy()


print(
    "删除前 Train Shape:",
    model_train.shape
)

print(
    "删除后 Train Shape:",
    model_train_clean.shape
)

print(
    "删除前 Test Shape:",
    model_test.shape
)

print(
    "删除后 Test Shape:",
    model_test_clean.shape
)

删除前 Train Shape: (307511, 451)
删除后 Train Shape: (307511, 441)
删除前 Test Shape: (48744, 450)
删除后 Test Shape: (48744, 440)


In [9]:
train_clean_feature_columns = [
    col
    for col in model_train_clean.columns
    if col != "TARGET"
]

print(
    "删除常量字段后 Train/Test 是否一致:",
    train_clean_feature_columns
    == model_test_clean.columns.tolist()
)

删除常量字段后 Train/Test 是否一致: True


In [10]:
# ============================================================
# 【读取步骤】
# 读取6张卫星特征表，取得真实字段范围
# ============================================================

bureau_features = pd.read_parquet(
    "data/processed/bureau_features.parquet"
)

bureau_balance_features = pd.read_parquet(
    "data/processed/bureau_balance_features.parquet"
)

previous_application_features = pd.read_parquet(
    "data/processed/previous_application_features.parquet"
)

installments_features = pd.read_parquet(
    "data/processed/installments_features.parquet"
)

pos_cash_features = pd.read_parquet(
    "data/processed/pos_cash_features.parquet"
)

credit_card_features = pd.read_parquet(
    "data/processed/credit_card_features.parquet"
)

In [11]:
# ============================================================
# 【建立配置】
# 对应每张卫星表、可用性标记和真实特征字段
# ============================================================

external_feature_groups = {
    "HAS_BUREAU_HISTORY": [
        col
        for col in bureau_features.columns
        if col != "SK_ID_CURR"
    ],

    "HAS_BUREAU_BALANCE_HISTORY": [
        col
        for col in bureau_balance_features.columns
        if col != "SK_ID_CURR"
    ],

    "HAS_PREVIOUS_APPLICATION_HISTORY": [
        col
        for col in previous_application_features.columns
        if col != "SK_ID_CURR"
    ],

    "HAS_INSTALLMENT_HISTORY": [
        col
        for col in installments_features.columns
        if col != "SK_ID_CURR"
    ],

    "HAS_POS_CASH_HISTORY": [
        col
        for col in pos_cash_features.columns
        if col != "SK_ID_CURR"
    ],

    "HAS_CREDIT_CARD_HISTORY": [
        col
        for col in credit_card_features.columns
        if col != "SK_ID_CURR"
    ]
}

In [12]:
for indicator_col, feature_columns in external_feature_groups.items():

    missing_in_train = [
        col
        for col in feature_columns
        if col not in model_train_clean.columns
    ]

    missing_in_test = [
        col
        for col in feature_columns
        if col not in model_test_clean.columns
    ]

    print(
        indicator_col,
        "字段数量:",
        len(feature_columns),
        "Train缺失字段:",
        missing_in_train,
        "Test缺失字段:",
        missing_in_test
    )

HAS_BUREAU_HISTORY 字段数量: 25 Train缺失字段: [] Test缺失字段: []
HAS_BUREAU_BALANCE_HISTORY 字段数量: 24 Train缺失字段: [] Test缺失字段: []
HAS_PREVIOUS_APPLICATION_HISTORY 字段数量: 56 Train缺失字段: [] Test缺失字段: []
HAS_INSTALLMENT_HISTORY 字段数量: 42 Train缺失字段: [] Test缺失字段: []
HAS_POS_CASH_HISTORY 字段数量: 50 Train缺失字段: [] Test缺失字段: []
HAS_CREDIT_CARD_HISTORY 字段数量: 80 Train缺失字段: [] Test缺失字段: []


In [13]:
# ============================================================
# 【清洗步骤 2】
# 对没有外部历史记录的客户，将对应整组特征填为0
# ============================================================

for indicator_col, feature_columns in external_feature_groups.items():

    # 1. 找出训练集中没有该类历史记录的客户
    train_no_history_mask = (
        model_train_clean[indicator_col] == 0
    )

    # 2. 只填这些客户对应特征组中的缺失值
    model_train_clean.loc[
        train_no_history_mask,
        feature_columns
    ] = (
        model_train_clean.loc[
            train_no_history_mask,
            feature_columns
        ]
        .fillna(0)
    )

    # 3. 找出测试集中没有该类历史记录的客户
    test_no_history_mask = (
        model_test_clean[indicator_col] == 0
    )

    # 4. 只填这些客户对应特征组中的缺失值
    model_test_clean.loc[
        test_no_history_mask,
        feature_columns
    ] = (
        model_test_clean.loc[
            test_no_history_mask,
            feature_columns
        ]
        .fillna(0)
    )

In [14]:
for indicator_col, feature_columns in external_feature_groups.items():

    train_no_history_mask = (
        model_train_clean[indicator_col] == 0
    )

    test_no_history_mask = (
        model_test_clean[indicator_col] == 0
    )

    train_remaining_missing = (
        model_train_clean.loc[
            train_no_history_mask,
            feature_columns
        ]
        .isna()
        .sum()
        .sum()
    )

    test_remaining_missing = (
        model_test_clean.loc[
            test_no_history_mask,
            feature_columns
        ]
        .isna()
        .sum()
        .sum()
    )

    print(
        indicator_col,
        "Train无历史客户剩余缺失:",
        train_remaining_missing,
        "Test无历史客户剩余缺失:",
        test_remaining_missing
    )

HAS_BUREAU_HISTORY Train无历史客户剩余缺失: 0 Test无历史客户剩余缺失: 0
HAS_BUREAU_BALANCE_HISTORY Train无历史客户剩余缺失: 0 Test无历史客户剩余缺失: 0
HAS_PREVIOUS_APPLICATION_HISTORY Train无历史客户剩余缺失: 0 Test无历史客户剩余缺失: 0
HAS_INSTALLMENT_HISTORY Train无历史客户剩余缺失: 0 Test无历史客户剩余缺失: 0
HAS_POS_CASH_HISTORY Train无历史客户剩余缺失: 0 Test无历史客户剩余缺失: 0
HAS_CREDIT_CARD_HISTORY Train无历史客户剩余缺失: 0 Test无历史客户剩余缺失: 0


In [15]:
remaining_missing_summary = (
    model_train_clean
    .drop(columns=["TARGET"])
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("remaining_missing_rate")
)

remaining_missing_summary[
    remaining_missing_summary[
        "remaining_missing_rate"
    ] > 0
].head(30)

,remaining_missing_rate
COMMONAREA_MODE,0.698723
COMMONAREA_AVG,0.698723
COMMONAREA_MEDI,0.698723
NONLIVINGAPARTMENTS_MEDI,0.694330
NONLIVINGAPARTMENTS_MODE,0.694330
NONLIVINGAPARTMENTS_AVG,0.694330
LIVINGAPARTMENTS_MEDI,0.683550
LIVINGAPARTMENTS_MODE,0.683550
LIVINGAPARTMENTS_AVG,0.683550
FLOORSMIN_MODE,0.678486


In [16]:
# ============================================================
# 【检查步骤】
# 查找剩余的类别字段缺失
# ============================================================

train_categorical_columns = (
    model_train_clean
    .drop(columns=["TARGET"])
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

test_categorical_columns = (
    model_test_clean
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)


print(
    "训练集类别字段数量:",
    len(train_categorical_columns)
)

print(
    "测试集类别字段数量:",
    len(test_categorical_columns)
)

print(
    "Train/Test 类别字段是否一致:",
    train_categorical_columns
    == test_categorical_columns
)


categorical_missing_summary = (
    model_train_clean[
        train_categorical_columns
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

categorical_missing_summary[
    categorical_missing_summary > 0
]

训练集类别字段数量: 24
测试集类别字段数量: 24
Train/Test 类别字段是否一致: True


EMPLOYMENT_TO_AGE_GROUP     55374
CREDIT_TO_GOODS_GROUP         278
ANNUITY_PER_PERSON_GROUP       14
REMAINING_INCOME_GROUP         12
ANNUITY_TO_INCOME_GROUP        12
CREDIT_PER_PERSON_GROUP         2
INCOME_PER_PERSON_GROUP         2
dtype: int64

In [17]:
# ============================================================
# 【清洗步骤 3】
# 将剩余类别特征缺失统一标记为 Missing
# ============================================================

for col in train_categorical_columns:

    # 1. 如果是 category 类型，先加入 Missing 类别
    if isinstance(
        model_train_clean[col].dtype,
        pd.CategoricalDtype
    ):

        if "Missing" not in model_train_clean[col].cat.categories:
            model_train_clean[col] = (
                model_train_clean[col]
                .cat.add_categories(["Missing"])
            )

        if "Missing" not in model_test_clean[col].cat.categories:
            model_test_clean[col] = (
                model_test_clean[col]
                .cat.add_categories(["Missing"])
            )

    # 2. 同时填充训练集和测试集
    model_train_clean[col] = (
        model_train_clean[col]
        .fillna("Missing")
    )

    model_test_clean[col] = (
        model_test_clean[col]
        .fillna("Missing")
    )


# 3. 检查类别字段剩余缺失
print(
    "训练集剩余类别缺失数量:",
    model_train_clean[
        train_categorical_columns
    ].isna().sum().sum()
)

print(
    "测试集剩余类别缺失数量:",
    model_test_clean[
        test_categorical_columns
    ].isna().sum().sum()
)

训练集剩余类别缺失数量: 0
测试集剩余类别缺失数量: 0


In [18]:
# ============================================================
# 【清洗步骤 4】
# 使用训练集中位数填补剩余数值缺失
# ============================================================


# 1. 找出训练集中的数值特征
numeric_feature_columns = (
    model_train_clean
    .drop(columns=["TARGET"])
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)


# 2. 排除客户编号
numeric_feature_columns = [
    col
    for col in numeric_feature_columns
    if col != "SK_ID_CURR"
]


# 3. 找出 Train 或 Test 中仍然存在缺失的数值字段
numeric_columns_with_missing = [
    col
    for col in numeric_feature_columns
    if (
        model_train_clean[col].isna().any()
        or model_test_clean[col].isna().any()
    )
]


print(
    "存在缺失的数值字段数量:",
    len(numeric_columns_with_missing)
)


# 4. 只使用训练集计算中位数
numeric_fill_values = (
    model_train_clean[
        numeric_columns_with_missing
    ]
    .median()
)


# 5. 检查是否有整列缺失，导致无法计算中位数
all_missing_numeric_columns = (
    numeric_fill_values[
        numeric_fill_values.isna()
    ]
    .index
    .tolist()
)


print(
    "无法计算训练集中位数的字段:",
    all_missing_numeric_columns
)

存在缺失的数值字段数量: 151
无法计算训练集中位数的字段: []


In [19]:
# ============================================================
# 【填补步骤】
# 用训练集的中位数同时处理 Train 和 Test
# ============================================================


# 1. 填补训练集
model_train_clean[
    numeric_columns_with_missing
] = (
    model_train_clean[
        numeric_columns_with_missing
    ]
    .fillna(numeric_fill_values)
)


# 2. 使用完全相同的中位数填补测试集
model_test_clean[
    numeric_columns_with_missing
] = (
    model_test_clean[
        numeric_columns_with_missing
    ]
    .fillna(numeric_fill_values)
)

In [20]:
# ============================================================
# 【最终检查步骤】
# 检查缺失值、Inf、主键和字段一致性
# ============================================================


# 1. 检查剩余缺失值
train_remaining_missing = (
    model_train_clean
    .drop(columns=["TARGET"])
    .isna()
    .sum()
    .sum()
)

test_remaining_missing = (
    model_test_clean
    .isna()
    .sum()
    .sum()
)


print(
    "训练集最终剩余缺失数量:",
    train_remaining_missing
)

print(
    "测试集最终剩余缺失数量:",
    test_remaining_missing
)


# 2. 检查 Inf
train_inf_count = (
    np.isinf(
        model_train_clean
        .drop(columns=["TARGET"])
        .select_dtypes(include=["number"])
    )
    .sum()
    .sum()
)

test_inf_count = (
    np.isinf(
        model_test_clean
        .select_dtypes(include=["number"])
    )
    .sum()
    .sum()
)


print(
    "训练集 Inf 数量:",
    train_inf_count
)

print(
    "测试集 Inf 数量:",
    test_inf_count
)


# 3. 检查客户唯一性
print(
    "训练集重复客户数量:",
    model_train_clean[
        "SK_ID_CURR"
    ].duplicated().sum()
)

print(
    "测试集重复客户数量:",
    model_test_clean[
        "SK_ID_CURR"
    ].duplicated().sum()
)


# 4. 检查 Train/Test 字段
train_clean_feature_columns = [
    col
    for col in model_train_clean.columns
    if col != "TARGET"
]

print(
    "最终 Train/Test 字段是否一致:",
    train_clean_feature_columns
    == model_test_clean.columns.tolist()
)


# 5. 查看最终 Shape
print(
    "最终训练集 Shape:",
    model_train_clean.shape
)

print(
    "最终测试集 Shape:",
    model_test_clean.shape
)

训练集最终剩余缺失数量: 0
测试集最终剩余缺失数量: 0
训练集 Inf 数量: 0
测试集 Inf 数量: 0
训练集重复客户数量: 0
测试集重复客户数量: 0
最终 Train/Test 字段是否一致: True
最终训练集 Shape: (307511, 441)
最终测试集 Shape: (48744, 440)


In [21]:
from pathlib import Path


# ============================================================
# 【保存结果】
# 保存最终清洗后的建模数据
# ============================================================


# 1. 建立保存路径
train_clean_path = Path(
    "data/processed/model_train_clean.parquet"
)

test_clean_path = Path(
    "data/processed/model_test_clean.parquet"
)


# 2. 保存训练集
model_train_clean.to_parquet(
    train_clean_path,
    index=False
)


# 3. 保存测试集
model_test_clean.to_parquet(
    test_clean_path,
    index=False
)


# 4. 检查文件
print(
    "训练集文件是否存在:",
    train_clean_path.exists()
)

print(
    "测试集文件是否存在:",
    test_clean_path.exists()
)

print(
    "训练集保存位置:",
    train_clean_path.resolve()
)

print(
    "测试集保存位置:",
    test_clean_path.resolve()
)

训练集文件是否存在: True
测试集文件是否存在: True
训练集保存位置: /Users/hulk/Documents/面试项目/credit-risk-portfolio-analytics/data/processed/model_train_clean.parquet
测试集保存位置: /Users/hulk/Documents/面试项目/credit-risk-portfolio-analytics/data/processed/model_test_clean.parquet
